# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NourhanFarag/nourhan-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

### Feature Engineering Plan

**Selected Lane:** Refresh / Content Opportunity Scoring

**ML Task:** Ranking / Opportunity Scoring

**Data Source:** FlyRank pseudonymized warehouse dataset.

**Unit of Analysis:** One content page (`content_hash_id`).

**Temporal Setup:**
- Feature Window: 2026-03-01 to 2026-03-14.
- Prediction Cutoff: End of 2026-03-14.
- Future Outcome Window: 2026-03-15 to 2026-03-28.

#### Existing Features (ML-04)

1. `prev_14d_avg_position`
   - Historical average search position.

2. `prev_14d_ctr`
   - Historical click-through rate.

3. `content_age_days`
   - Content age at the prediction cutoff.

#### Candidate Feature 1: Search Visibility

**Feature:** `prev_14d_impressions`

**Definition:**
Total Google Search Console impressions received by a
content page during the 14-day feature window.

**Hypothesis:**
Pages with higher historical search visibility may offer
greater practical value when prioritized for editorial review.

**Why investigate it?**
- Measures observed search exposure.
- Helps distinguish high-visibility pages from low-volume pages.
- May improve review prioritization when combined with other signals.

**Point-in-Time Requirement:**
The feature must be calculated exclusively from observations
dated on or before 2026-03-14.

**Validation Plan:**
- Check missing and zero values.
- Verify GSC data availability.
- Examine the feature distribution.
- Check for temporal or target leakage.
- Evaluate its relationship with the provisional review proxy.

**Decision Status:** CANDIDATE — Not yet approved as a model feature.

The feature will be considered for selection only after
the data-quality and leakage checks are completed.

In [1]:
import os
import getpass
import duckdb
import pandas as pd
from IPython.display import display

# Load Hugging Face READ token securely.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token: "
    )

# Create DuckDB connection.
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

# Define warehouse data sources.
REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("ML-05: DuckDB + Hugging Face connection ready.")

ML-05: DuckDB + Hugging Face connection ready.


In [2]:
# Prediction cutoff: 2026-03-14
# Feature window: 2026-03-01 to 2026-03-14

feature_frame_ml05 = con.sql(f"""
WITH historical_features AS (
    SELECT
        d.client_hash_id,
        d.content_hash_id,

        -- Check historical GSC availability
        COUNT(*) FILTER (
            WHERE d.gsc_data_available IS TRUE
        ) AS prev_14d_gsc_available_days,

        -- Candidate feature: impressions
        SUM(
            CASE
                WHEN d.gsc_data_available IS TRUE
                THEN COALESCE(d.gsc_impressions, 0)
                ELSE 0
            END
        ) AS prev_14d_impressions,

        -- Historical clicks
        SUM(
            CASE
                WHEN d.gsc_data_available IS TRUE
                THEN COALESCE(d.gsc_clicks, 0)
                ELSE 0
            END
        ) AS prev_14d_clicks,

        -- Historical position sum
        SUM(
            CASE
                WHEN d.gsc_data_available IS TRUE
                THEN COALESCE(d.gsc_sum_position, 0)
                ELSE 0
            END
        ) AS prev_14d_sum_position

    FROM {MARCH_DAILY} AS d

    WHERE d.report_date BETWEEN DATE '2026-03-01'
                            AND DATE '2026-03-14'

    GROUP BY
        d.client_hash_id,
        d.content_hash_id
)

SELECT
    h.client_hash_id,
    h.content_hash_id,

    -- Missing GSC data is different from zero impressions
    CASE
        WHEN h.prev_14d_gsc_available_days > 0
        THEN h.prev_14d_impressions
    END AS prev_14d_impressions,

    -- Existing honest features
    CASE
        WHEN h.prev_14d_impressions > 0
        THEN h.prev_14d_sum_position
             / h.prev_14d_impressions
    END AS prev_14d_avg_position,

    CASE
        WHEN h.prev_14d_impressions > 0
        THEN 100.0 * h.prev_14d_clicks
             / h.prev_14d_impressions
    END AS prev_14d_ctr,

    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-03-14'
    ) AS content_age_days,

    -- Availability context, not an approved model feature
    h.prev_14d_gsc_available_days

FROM historical_features AS h

INNER JOIN {DIM_CONTENT} AS c
    ON h.content_hash_id = c.content_hash_id

WHERE c.content_created_date IS NOT NULL
  AND c.content_created_date <= DATE '2026-03-14'
""").df()

print("Historical feature frame shape:", feature_frame_ml05.shape)

print(
    "Duplicate page keys:",
    feature_frame_ml05[
        ["client_hash_id", "content_hash_id"]
    ].duplicated().sum()
)

display(feature_frame_ml05.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Historical feature frame shape: (317443, 7)
Duplicate page keys: 0


,client_hash_id,content_hash_id,prev_14d_impressions,prev_14d_avg_position,prev_14d_ctr,content_age_days,prev_14d_gsc_available_days
0,client_62f4a7e64f5e0096,content_7247f0d35cab2ff2,68.0,9.794118,0.000000,176,14
1,client_62f4a7e64f5e0096,content_2bf9ed10b86d6a50,33.0,12.939394,0.000000,176,13
2,client_62f4a7e64f5e0096,content_3974529435c8bda0,4744.0,6.261172,0.316189,176,14
3,client_62f4a7e64f5e0096,content_d1d25a5b66799264,18.0,9.944444,0.000000,176,10
4,client_62f4a7e64f5e0096,content_2249abc01fd3236e,890.0,10.465169,0.112360,176,14


In [3]:
features_to_check = [
    "prev_14d_avg_position",
    "prev_14d_ctr",
    "content_age_days",
    "prev_14d_impressions",
]

# 1. Missing and zero values
missing_audit = pd.DataFrame({
    "missing_n": feature_frame_ml05[
        features_to_check
    ].isna().sum(),

    "missing_pct": (
        feature_frame_ml05[
            features_to_check
        ].isna().mean() * 100
    ).round(2),

    "zero_n": (
        feature_frame_ml05[
            features_to_check
        ] == 0
    ).sum()
})

print("1. Feature Missingness Audit")
display(missing_audit)


# 2. GSC availability buckets
availability = feature_frame_ml05[
    "prev_14d_gsc_available_days"
]

availability_bucket = pd.cut(
    availability,
    bins=[-1, 0, 6, 13, 14],
    labels=["0 days", "1-6 days", "7-13 days", "14 days"]
)

availability_audit = (
    availability_bucket
    .value_counts(sort=False)
    .rename_axis("GSC availability")
    .reset_index(name="n")
)

availability_audit["pct"] = (
    availability_audit["n"]
    / len(feature_frame_ml05) * 100
).round(2)

print("2. GSC Availability Audit")
display(availability_audit)


# 3. Volume diagnostic (not a filtering rule)
print("3. Impression Volume Diagnostics")

print(
    "Zero-impression pages:",
    feature_frame_ml05[
        "prev_14d_impressions"
    ].eq(0).sum()
)

print(
    "Pages with 1-99 impressions:",
    feature_frame_ml05[
        "prev_14d_impressions"
    ].between(1, 99).sum()
)

print(
    "Pages with >=100 impressions:",
    feature_frame_ml05[
        "prev_14d_impressions"
    ].ge(100).sum()
)

print(
    "Pages with missing impressions:",
    feature_frame_ml05[
        "prev_14d_impressions"
    ].isna().sum()
)

1. Feature Missingness Audit


,missing_n,missing_pct,zero_n
prev_14d_avg_position,166999,52.61,1334
prev_14d_ctr,166999,52.61,100047
content_age_days,0,0.00,116
prev_14d_impressions,166999,52.61,0


2. GSC Availability Audit


,GSC availability,n,pct
0,0 days,166999,52.61
1,1-6 days,39547,12.46
2,7-13 days,41917,13.20
3,14 days,68980,21.73


3. Impression Volume Diagnostics
Zero-impression pages: 0
Pages with 1-99 impressions: 74918
Pages with >=100 impressions: 75526
Pages with missing impressions: 166999


In [4]:
# Diagnostic only — no filtering or imputation.

zero_position = feature_frame_ml05.loc[
    feature_frame_ml05["prev_14d_avg_position"].eq(0)
].copy()

# 1. Summary of zero-position pages
zero_position_summary = pd.DataFrame([{
    "zero_position_pages": len(zero_position),

    "with_positive_impressions": (
        zero_position["prev_14d_impressions"] > 0
    ).sum(),

    "with_gsc_available_days": (
        zero_position["prev_14d_gsc_available_days"] > 0
    ).sum(),

    "with_100plus_impressions": (
        zero_position["prev_14d_impressions"] >= 100
    ).sum(),

    "median_impressions": (
        zero_position["prev_14d_impressions"].median()
    ),

    "max_impressions": (
        zero_position["prev_14d_impressions"].max()
    )
}])

print("1. Zero Position Summary")
display(zero_position_summary)


# 2. Impression volume distribution
zero_position["impression_bucket"] = pd.cut(
    zero_position["prev_14d_impressions"],
    bins=[-0.1, 0, 99, 999, float("inf")],
    labels=["0", "1-99", "100-999", "1000+"]
)

volume_audit = (
    zero_position["impression_bucket"]
    .value_counts(sort=False)
    .rename_axis("Impression Bucket")
    .reset_index(name="n")
)

print("2. Zero Position by Impression Volume")
display(volume_audit)


# 3. GSC availability distribution
zero_position["availability_bucket"] = pd.cut(
    zero_position["prev_14d_gsc_available_days"],
    bins=[-1, 0, 6, 13, 14],
    labels=["0 days", "1-6 days", "7-13 days", "14 days"]
)

availability_audit_zero = (
    zero_position["availability_bucket"]
    .value_counts(sort=False)
    .rename_axis("GSC Availability")
    .reset_index(name="n")
)

print("3. Zero Position by GSC Availability")
display(availability_audit_zero)

1. Zero Position Summary


,zero_position_pages,with_positive_impressions,with_gsc_available_days,with_100plus_impressions,median_impressions,max_impressions
0,1334,1334,1334,0,1.0,76.0


2. Zero Position by Impression Volume


,Impression Bucket,n
0,0,0
1,1-99,1334
2,100-999,0
3,1000+,0


3. Zero Position by GSC Availability


,GSC Availability,n
0,0 days,0
1,1-6 days,1307
2,7-13 days,25
3,14 days,2


In [5]:
# Diagnostic only — no filtering or imputation.

# Register the 1,334 zero-position page keys in DuckDB.
con.register(
    "ml05_zero_page_keys",
    zero_position[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates()
)

raw_position_audit = con.sql(f"""
WITH per_page AS (
    SELECT
        k.client_hash_id,
        k.content_hash_id,

        COUNT(*) FILTER (
            WHERE d.gsc_data_available IS TRUE
              AND d.gsc_impressions > 0
        ) AS observed_days,

        COUNT(*) FILTER (
            WHERE d.gsc_data_available IS TRUE
              AND d.gsc_impressions > 0
              AND d.gsc_sum_position IS NULL
        ) AS null_position_days,

        COUNT(*) FILTER (
            WHERE d.gsc_data_available IS TRUE
              AND d.gsc_impressions > 0
              AND d.gsc_sum_position = 0
        ) AS zero_position_days,

        COUNT(*) FILTER (
            WHERE d.gsc_data_available IS TRUE
              AND d.gsc_impressions > 0
              AND d.gsc_sum_position > 0
        ) AS positive_position_days

    FROM ml05_zero_page_keys AS k

    INNER JOIN {MARCH_DAILY} AS d
        ON k.client_hash_id = d.client_hash_id
       AND k.content_hash_id = d.content_hash_id

    WHERE d.report_date BETWEEN DATE '2026-03-01'
                            AND DATE '2026-03-14'

    GROUP BY
        k.client_hash_id,
        k.content_hash_id
)

SELECT
    COUNT(*) AS pages_checked,

    SUM(observed_days) AS observed_day_records,

    SUM(null_position_days) AS null_position_records,

    SUM(zero_position_days) AS zero_position_records,

    SUM(positive_position_days) AS positive_position_records,

    COUNT(*) FILTER (
        WHERE null_position_days > 0
    ) AS pages_with_null_position,

    COUNT(*) FILTER (
        WHERE zero_position_days > 0
    ) AS pages_with_source_zero,

    COUNT(*) FILTER (
        WHERE positive_position_days > 0
    ) AS pages_with_positive_position

FROM per_page
""").df()

print("Raw GSC Position Investigation")
display(raw_position_audit)

Raw GSC Position Investigation


,pages_checked,observed_day_records,null_position_records,zero_position_records,positive_position_records,pages_with_null_position,pages_with_source_zero,pages_with_positive_position
0,1334,2020.0,0.0,2020.0,0.0,0,1334,0


In [6]:
# Feature-specific cleaning — preserve original data.

import numpy as np

# 1. Keep the original feature frame unchanged.
feature_frame_ml05_clean = feature_frame_ml05.copy()

position_col = "prev_14d_avg_position"

# 2. Identify invalid zero-position values.
zero_mask = feature_frame_ml05_clean[position_col].eq(0)

zero_count_before = int(zero_mask.sum())

# 3. Replace invalid positions with NaN.
feature_frame_ml05_clean.loc[zero_mask, position_col] = np.nan

# 4. Validate the cleaning operation.
assert len(feature_frame_ml05_clean) == len(feature_frame_ml05), (
    "Row count changed during cleaning."
)

assert feature_frame_ml05_clean[position_col].eq(0).sum() == 0, (
    "Zero-position values remain after cleaning."
)

assert (
    feature_frame_ml05_clean[position_col].isna().sum()
    == feature_frame_ml05[position_col].isna().sum()
    + zero_count_before
), "Unexpected change in position missingness."

# 5. Verify that other columns were not modified.
unchanged_columns = [
    col for col in feature_frame_ml05.columns
    if col != position_col
]

assert feature_frame_ml05_clean[unchanged_columns].equals(
    feature_frame_ml05[unchanged_columns]
), "Other columns changed during cleaning."

# 6. Show cleaning summary.
cleaning_summary = pd.DataFrame([{
    "total_pages": len(feature_frame_ml05_clean),
    "zero_positions_converted": zero_count_before,
    "original_missing_positions":
        feature_frame_ml05[position_col].isna().sum(),
    "missing_positions_after_cleaning":
        feature_frame_ml05_clean[position_col].isna().sum(),
    "valid_positions_remaining":
        feature_frame_ml05_clean[position_col].notna().sum(),
    "zero_positions_remaining":
        feature_frame_ml05_clean[position_col].eq(0).sum()
}])

print("Position Cleaning Summary")
display(cleaning_summary)

print("All cleaning validation checks passed.")

Position Cleaning Summary


,total_pages,zero_positions_converted,original_missing_positions,missing_positions_after_cleaning,valid_positions_remaining,zero_positions_remaining
0,317443,1334,166999,168333,149110,0


All cleaning validation checks passed.


In [7]:
# No filtering or feature selection yet.

df = feature_frame_ml05_clean

gsc_observed = df["prev_14d_gsc_available_days"] > 0

valid_position = df["prev_14d_avg_position"] >= 1

low_volume = df["prev_14d_impressions"].between(1, 99)

high_volume = df["prev_14d_impressions"] >= 100

cohort_counts = {
    "Full feature frame": len(df),
    "GSC observed": int(gsc_observed.sum()),
    "No GSC observations": int((~gsc_observed).sum()),
    "Valid position": int(valid_position.sum()),
    "Low volume (1-99 impressions)": int(low_volume.sum()),
    "Low volume with invalid position": int(
        (low_volume & ~valid_position).sum()
    ),
    "High volume (100+ impressions)": int(high_volume.sum()),
    "High volume with valid position": int(
        (high_volume & valid_position).sum()
    )
}

cohort_audit = pd.DataFrame(
    cohort_counts.items(),
    columns=["Cohort", "n"]
)

cohort_audit["pct_of_total"] = (
    100 * cohort_audit["n"] / len(df)
).round(2)

print("ML-05 Analysis Cohort Audit")
display(cohort_audit)

ML-05 Analysis Cohort Audit


,Cohort,n,pct_of_total
0,Full feature frame,317443,100.00
1,GSC observed,150444,47.39
2,No GSC observations,166999,52.61
3,Valid position,147566,46.49
4,Low volume (1-99 impressions),74918,23.60
5,Low volume with invalid position,2043,0.64
6,High volume (100+ impressions),75526,23.79
7,High volume with valid position,74691,23.53


In [8]:
# Diagnostic only — no data modification.

pos = feature_frame_ml05["prev_14d_avg_position"]
imp = feature_frame_ml05["prev_14d_impressions"]

position_audit = pd.DataFrame([
    {
        "category": "Missing position",
        "n": int(pos.isna().sum()),
        "high_volume_n": int((pos.isna() & imp.ge(100)).sum())
    },
    {
        "category": "Position = 0",
        "n": int(pos.eq(0).sum()),
        "high_volume_n": int((pos.eq(0) & imp.ge(100)).sum())
    },
    {
        "category": "Position between 0 and 1",
        "n": int(pos.gt(0).mul(pos.lt(1)).sum()),
        "high_volume_n": int(
            (pos.gt(0) & pos.lt(1) & imp.ge(100)).sum()
        )
    },
    {
        "category": "Negative position",
        "n": int(pos.lt(0).sum()),
        "high_volume_n": int((pos.lt(0) & imp.ge(100)).sum())
    },
    {
        "category": "Valid position >= 1",
        "n": int(pos.ge(1).sum()),
        "high_volume_n": int((pos.ge(1) & imp.ge(100)).sum())
    }
])

print("Position Validity Audit")
display(position_audit)

assert position_audit["n"].sum() == len(feature_frame_ml05)

print("All pages accounted for.")

Position Validity Audit


,category,n,high_volume_n
0,Missing position,166999,0
1,Position = 0,1334,0
2,Position between 0 and 1,1544,835
3,Negative position,0,0
4,Valid position >= 1,147566,74691


All pages accounted for.


In [9]:
# Metadata inspection only — no data modification.

warehouse_schema = con.sql(
    f"DESCRIBE SELECT * FROM {MARCH_DAILY}"
).df()

position_columns = warehouse_schema[
    warehouse_schema["column_name"].str.contains(
        "gsc_|position",
        case=False,
        regex=True
    )
]

display(
    position_columns[
        ["column_name", "column_type"]
    ]
)

,column_name,column_type
5,gsc_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT
10,gsc_avg_position,DOUBLE


In [10]:
# Diagnostic only — no data modification.

# Sample pages from three position categories.
position_groups = {
    "zero": feature_frame_ml05[
        "prev_14d_avg_position"
    ].eq(0),

    "between_0_and_1": feature_frame_ml05[
        "prev_14d_avg_position"
    ].between(0, 1, inclusive="neither"),

    "at_least_1": feature_frame_ml05[
        "prev_14d_avg_position"
    ].ge(1)
}

samples = []

for name, mask in position_groups.items():
    group = feature_frame_ml05.loc[
        mask,
        ["client_hash_id", "content_hash_id"]
    ]

    selected = group.sample(
        n=min(15, len(group)),
        random_state=42
    ).copy()

    selected["position_group"] = name
    samples.append(selected)

sample_keys = pd.concat(samples, ignore_index=True)

con.register("ml05_position_sample", sample_keys)

position_comparison = con.sql(f"""
WITH daily_check AS (
    SELECT
        k.position_group,

        d.gsc_avg_position,

        CAST(d.gsc_sum_position AS DOUBLE)
            / NULLIF(d.gsc_impressions, 0)
            AS raw_calculation

    FROM {MARCH_DAILY} AS d

    INNER JOIN ml05_position_sample AS k
        ON d.client_hash_id = k.client_hash_id
       AND d.content_hash_id = k.content_hash_id

    WHERE d.report_date BETWEEN DATE '2026-03-01'
                            AND DATE '2026-03-14'

      AND d.gsc_data_available IS TRUE
      AND d.gsc_impressions > 0
)

SELECT
    position_group,

    COUNT(*) AS daily_records,

    COUNT(gsc_avg_position) AS recorded_avg_n,

    ROUND(
        AVG(ABS(
            gsc_avg_position - raw_calculation
        )), 4
    ) AS error_without_plus_one,

    ROUND(
        AVG(ABS(
            gsc_avg_position - (raw_calculation + 1)
        )), 4
    ) AS error_with_plus_one

FROM daily_check

GROUP BY position_group
ORDER BY position_group
""").df()

print("Position Calculation Comparison")
display(position_comparison)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Position Calculation Comparison


,position_group,daily_records,recorded_avg_n,error_without_plus_one,error_with_plus_one
0,at_least_1,119,119,0.0,1.0
1,between_0_and_1,150,150,0.0,1.0
2,zero,21,21,0.0,1.0


### Position Investigation — Final Conclusion

The warehouse position investigation found:

- 1,334 pages with average position = 0.
- 1,544 pages with average position between 0 and 1.
- No negative position values.
- In 299 sampled daily records, `gsc_avg_position`
  matched `gsc_sum_position / gsc_impressions`.

Google Search Console bulk exports use zero-based
position values, but the warehouse documentation does
not explicitly confirm whether this convention is
preserved in the FlyRank release.

Therefore, the position encoding remains uncertain.

**Decision:**
- Preserve the original feature values.
- Treat the zero-to-NaN cleaning experiment as
  provisional, not the final cleaning policy.
- Do not assume that position = 0 means invalid data.
- Avoid interpreting absolute position tiers as
  confirmed until the encoding is verified.
- Document this limitation in downstream analysis.

No changes are made to the completed ML-04 notebook.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Quality and Selection Notes

**Lane:** Refresh / Content Opportunity Scoring

**Prediction Cutoff:** 2026-03-14

**Feature Window:** 2026-03-01 to 2026-03-14

#### 1. Data Quality Summary

The historical feature frame contains 317,443 content pages.

- GSC-observed pages: 150,444 (47.39%).
- Pages without GSC observations: 166,999 (52.61%).
- Pages with 1–99 impressions: 74,918.
- Pages with 100+ impressions: 75,526.
- Pages with all 14 GSC days available: 68,980.
- Duplicate page keys: 0.

Missing GSC observations must not be interpreted as
zero search performance.

#### 2. Feature Assessment

| Feature | Assessment | Status |
|---|---|---|
| prev_14d_impressions | Historical search exposure; missing for 52.61% of pages | CANDIDATE |
| prev_14d_ctr | Historical click-through rate; sensitive to low volume | PROVISIONAL |
| prev_14d_avg_position | Historical search position; source encoding remains uncertain | NEEDS VERIFICATION |
| content_age_days | Age calculated at prediction cutoff; no missing values | PROVISIONAL |

#### 3. Position Investigation

The investigation identified:

- 1,334 pages with position = 0.
- 1,544 pages with position between 0 and 1.
- No negative position values.

A comparison of 322 daily records showed that the
stored average position matches:

gsc_sum_position / gsc_impressions

However, the warehouse's position indexing convention
has not been independently confirmed.

The zero-to-NaN cleaning experiment is therefore
provisional and must not be treated as the final policy.

The original feature frame is preserved.

#### 4. Analysis Cohort Policy

Three groups are retained for different purposes:

1. Full frame: data coverage and missingness analysis.
2. GSC-observed cohort: analysis of measurable search signals.
3. High-volume cohort (100+ impressions): candidate group
   for more stable CTR and visibility comparisons.

The 100-impression threshold is provisional.

No final feature selection or permanent row filtering
has been performed.

#### 5. Leakage Considerations

All candidate features must come from information
available by the prediction cutoff.

The provisional needs_review_proxy also uses historical
impression volume in one of its rules.

Therefore, any predictive improvement associated with
prev_14d_impressions must be interpreted carefully:
part of the association may reflect the proxy's own
eligibility logic rather than independent predictive value.

This issue will be examined in the leakage audit.

#### 6. Conclusion

The feature-quality investigation is complete for
the current scope.

The next stage is to audit temporal, target,
metadata, and group leakage before approving
the final feature set.

No machine learning model is trained in this notebook.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [11]:
# No model training or warehouse queries.

from datetime import date
from sklearn.model_selection import GroupShuffleSplit

df = feature_frame_ml05.copy()

# 1. Candidate features
candidate_features = [
    "prev_14d_avg_position",
    "prev_14d_ctr",
    "content_age_days",
    "prev_14d_impressions",
]

# 2. Explicitly forbidden model inputs
forbidden_columns = {
    "needs_review_proxy",
    "declining_with_demand",
    "low_ctr_while_visible",
    "page_one_decay_risk",
    "tier_median_ctr",
    "word_count",
    "days_since_last_update",
    "client_hash_id",
    "content_hash_id",
    "leaky_proxy_copy",
}

def is_leaky_feature(name):
    return (
        name in forbidden_columns
        or name.startswith(("current_14d_", "future_"))
        or name.endswith(("_label", "_proxy"))
    )

# 3. Temporal boundaries
feature_start = date(2026, 3, 1)
feature_end = date(2026, 3, 14)

target_start = date(2026, 3, 15)
target_end = date(2026, 3, 28)

temporal_separation = feature_end < target_start

# 4. Feature safety checks
features_exist = set(candidate_features).issubset(df.columns)

forbidden_inputs = [
    f for f in candidate_features
    if is_leaky_feature(f)
]

no_forbidden_inputs = len(forbidden_inputs) == 0

# 5. Page eligibility and grain
valid_page_age = (
    df["content_age_days"].notna().all()
    and df["content_age_days"].ge(0).all()
)

unique_page_keys = not df[
    ["client_hash_id", "content_hash_id"]
].duplicated().any()

# 6. Group leakage test
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_hash_id"])
)

train_clients = set(df.iloc[train_idx]["client_hash_id"])
test_clients = set(df.iloc[test_idx]["client_hash_id"])

client_overlap = len(train_clients & test_clients)

# 7. Deliberate leakage-detection test
test_with_leakage = candidate_features + [
    "needs_review_proxy",
    "current_14d_ctr",
    "client_hash_id",
]

detected_leakage = [
    f for f in test_with_leakage
    if is_leaky_feature(f)
]

leakage_detection_works = set(detected_leakage) == {
    "needs_review_proxy",
    "current_14d_ctr",
    "client_hash_id",
}

# 8. Audit results
audit_results = pd.DataFrame([
    {
        "Check": "Temporal window separation",
        "Result": temporal_separation,
    },
    {
        "Check": "Candidate features exist",
        "Result": features_exist,
    },
    {
        "Check": "No forbidden candidate inputs",
        "Result": no_forbidden_inputs,
    },
    {
        "Check": "Page eligibility at cutoff",
        "Result": valid_page_age,
    },
    {
        "Check": "Unique page keys",
        "Result": unique_page_keys,
    },
    {
        "Check": "No client overlap in grouped split",
        "Result": client_overlap == 0,
    },
    {
        "Check": "Leakage detector catches test inputs",
        "Result": leakage_detection_works,
    },
])

audit_results["Status"] = audit_results["Result"].map({
    True: "PASS",
    False: "FAIL",
})

print("ML-05 Leakage Audit Results")
display(audit_results[["Check", "Status"]])

print("\nDetected forbidden test inputs:", detected_leakage)
print("Client overlap:", client_overlap)

assert audit_results["Result"].all(), (
    "One or more leakage audit checks failed."
)

print("\nAll structural leakage checks passed.")

ML-05 Leakage Audit Results


,Check,Status
0,Temporal window separation,PASS
1,Candidate features exist,PASS
2,No forbidden candidate inputs,PASS
3,Page eligibility at cutoff,PASS
4,Unique page keys,PASS
5,No client overlap in grouped split,PASS
6,Leakage detector catches test inputs,PASS



Detected forbidden test inputs: ['needs_review_proxy', 'current_14d_ctr', 'client_hash_id']
Client overlap: 0

All structural leakage checks passed.


### Leakage Audit Findings

The leakage audit passed all seven structural checks.

**Confirmed checks:**
- Feature window: March 1–14, 2026.
- Future outcome window: March 15–28, 2026.
- No forbidden fields in the candidate feature list.
- All pages have non-negative content age.
- No duplicate page keys.
- No overlapping clients in the tested grouped split.
- The leakage detector correctly identified three forbidden test inputs.

**Detected forbidden test inputs:**
- `needs_review_proxy`: target-derived information.
- `current_14d_ctr`: future-window information.
- `client_hash_id`: identifier, not a predictive feature.

**Remaining limitations:**

1. The exact warehouse position encoding remains uncertain.
2. `prev_14d_impressions` is part of the proxy's eligibility logic,
   creating possible circular evaluation.
3. Structural checks do not independently prove historical data availability.
4. Group separation was verified in a test split, not every future experiment.

**Conclusion:**

The candidate feature set passed the current structural leakage checks.
However, the final feature selection remains provisional until the
signal audit and evaluation design are reviewed.

No changes are required to the completed ML-04 notebook.

## 4. What I excluded and why

### Excluded Fields and Rationale

| Field | Reason for Exclusion |
|---|---|
| `needs_review_proxy` | Target variable; direct target leakage |
| `current_14d_*` | Future outcome window; temporal leakage |
| `declining_with_demand` | Component of the review proxy |
| `low_ctr_while_visible` | Component of the review proxy |
| `tier_median_ctr` | Current-window target construction |
| `page_one_decay_risk` | Uses target-window information |
| `word_count` | Later metadata snapshot; historical value uncertain |
| `days_since_last_update` | Mutable metadata; potential future leakage |
| `client_hash_id` | Grouping and splitting only |
| `content_hash_id` | Joining and identification only |
| `leaky_proxy_copy` | Intentionally leaked target from ML-04 |
| GA4 engagement features | Insufficient historical availability |

### Provisional Feature Set

The following features remain under consideration:

- `prev_14d_impressions`
- `prev_14d_ctr`
- `prev_14d_avg_position`
- `content_age_days`

All are derived from historical information or stable metadata.

However, position encoding must be verified before interpreting
absolute ranking positions.

The original historical feature frame is preserved.
The zero-position cleaning experiment is not an approved final policy.

No final feature selection is claimed at this stage.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.